## Loading libraries

Loading the required libraries.

In [38]:
import sys
import pandas as pd
from collections import Counter
import json
import warnings
warnings.simplefilter(action='ignore')
from collections import Counter
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from tqdm.notebook import tqdm

## Settings
+ `clinical_file` : clincal EHRs source data file
+ `followup_file` : follow up table dump from db (used for BMI extraction from first followup)
+ `risk_file`: risk-factors table from db (used for successive BMI measure extraction)
+ `use_events` : the type list of events extracted form the clinical EHRs.
+ `infection_map_file` : the mapping (categorization) of infection events
+ `comorbidity_map_file` : the mapping (cathegorization) of comorbidities
+ `static_attributes_file`: list of static variables of EHR to be considred
+ `translator_file` : the file for english language traslation of all clinical terms
+ `stop_at_1st_inf`: `True` if sequence is truncated at 1st infection event
+ `flwup_nth` : number of follow ups for event sequence truncation.
+ `dataset_file`: dataset filename

NOTE: if you want to produce all events with no cut, set: `stop_at_1st_inf=False` and `flwup_nth=np.inf`

In [39]:
class Settings:
    suffix = "tutti_5_flwup"
    clinical_file = "data/file_aggiornato_analisi_14_04_26.csv"
    suffix = "14_04_26"
    followup_file = "files/follow_up_orig.csv"
    risk_file = "files/fattori_rischio_corr.csv"
    use_events = "splenectomizzato,interventi,terapie,vaccini_dosi,comorbidita,eventi_inf_raggruppati,eventi_tromb_raggruppati,followup,plt_flwup_numeriche"
    infection_map_file = "files/infezioni_mappa.json"
    comorbidity_map_file = "files/comorbidita_mappa.json"
    static_attributes_file = "files/static_vars.txt"
    translator_file = "files/translator_plus.json"
    icd_infections_file = "icd_files/icd10_infections_ccs.csv"
    icd_therapies_file = "icd_files/icd10pcs_therapies_ccs.csv"
    icd_vaccines_file = "icd_files/hcpcs_vaccines_ccs.csv"
    icd_diseases_file = "icd_files/icd10_diseases_ccs.csv"
    icd_procedures_file = "icd_files/icd10pcs_procedures_ccs.csv"
    stop_at_1st_inf = False           # False for no stop on 1st infection
    flwup_nth = np.inf                # np.inf for no cut
    dataset_file = f'dataset_{suffix}'
    events_file = f'events_{suffix}'
    data_file = f"dati_{suffix}"
    lang = 'EN'  # 'EN'
    with_qt = False
    with_timeshift = True
    with_bmichange = True

args = Settings()

## Data Preparation

### Loading clinical data
all null values are converted into pandas NA values

In [40]:
df_clinica = pd.read_csv(args.clinical_file, index_col=0)
df_clinica.replace({'NA': np.nan, 'nan': np.nan, '': np.nan, '<NA>': np.nan, 'NON SO': np.nan, "[ NON SO ]": np.nan}, inplace=True)
if args.lang == 'IT': 
    df_clinica['splenectomizzato'].replace({0: 'NO', 1: 'SI'}, inplace=True)
else: 
    df_clinica['splenectomizzato'].replace({0: 'NO', 1: 'YES'}, inplace=True)
    df_clinica.replace({'SI': 'YES'}, inplace=True)

comorbidity_map = json.load(open(args.comorbidity_map_file))
infection_map = json.load(open(args.infection_map_file))
df_clinica

,id_centro,sesso,data_nascita,tipo_osservazione,data_inizio_prospettica,area_pat_base,patologia_base,genotipo_beta1,genotipo_beta2,genotipo_alfa1,...,fumo,num_sigarette,anni_fumo,dislipidemia,colesterolo,trigliceridi,tsh,bmi,causa_del_decesso_tipologia,età_al_decesso
id,,,,,,,,,,,,,,,,,,,,,
1,9,M,1996-07-02,PROSPETTICO,01/03/2016,"SCD - (Sickle Cell Disease, Anemia/Malattia Fa...",( S/S ),ND,ND,ND,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,9,F,2001-04-15,PROSPETTICO,01/03/2016,Anemie emolitiche congenite,HS (Sferocitosi Ereditaria),ND,ND,ND,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,9,M,2006-12-19,PROSPETTICO,01/03/2016,Anemie emolitiche congenite,HS (Sferocitosi Ereditaria),ND,ND,ND,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,9,F,1999-09-14,PROSPETTICO,01/03/2016,Anemie emolitiche congenite,HS (Sferocitosi Ereditaria),ND,ND,ND,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,9,M,1995-02-28,PROSPETTICO,01/03/2016,Cause non emato-oncologiche,Traumi,ND,ND,ND,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1948,1,F,2015-04-26,RETRO/PROSP,27/06/2023,Anemie emolitiche congenite,HS (Sferocitosi Ereditaria),ND,ND,ND,...,2,NaN,NaN,2,NaN,NaN,NaN,22.8,NaN,NaN
1949,1,F,2016-12-27,RETRO/PROSP,12/07/2023,Anemie emolitiche congenite,HS (Sferocitosi Ereditaria),ND,ND,ND,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1950,1,F,2003-10-29,RETRO/PROSP,05/07/2023,Anemie emolitiche congenite,HS (Sferocitosi Ereditaria),ND,ND,ND,...,2,NaN,NaN,2,NaN,NaN,NaN,NaN,NaN,NaN


### Find real dead patients 

In [41]:
dead_indices = df_clinica[(df_clinica["data_decesso"] != "NaT") & (df_clinica["vivo"].isin(["YES", "ND", "NaN", np.nan]))].index.values
if args.lang == 'IT': 
    df_clinica.loc[dead_indices, 'vivo'] = "NO"
else:
    df_clinica.loc[dead_indices, 'vivo'] = "NO"
df_clinica['vivo'].value_counts()

vivo
YES    1474
ND      161
NO      153
Name: count, dtype: int64

### Loading ICD codes

In [42]:
icd_vaccines_df = pd.read_csv(args.icd_vaccines_file, index_col=0)
icd_infections_df = pd.read_csv(args.icd_infections_file, index_col=0)
icd_therapies_df = pd.read_csv(args.icd_therapies_file, index_col=0)
icd_diseases_df = pd.read_csv(args.icd_diseases_file, index_col=0)
icd_procedures_df = pd.read_csv(args.icd_procedures_file, index_col=0)

### Event sequences extraction
For all patients in the clinical db we extract (and sort) the list of events specified as the triple:
+ `sequences = [(<event-name>, <event-date><> <event-type>), ... ]`

NOTE: platelet change events are inserted only if different in range value (`low`, `medium`, `high`)

In [43]:
use_events = [v.strip() for v in args.use_events.split(",")]
for e in use_events:
    print(e)
    if e not in df_clinica.columns and e != 'followup':
        raise Exception(f"Event '{e}' must be in CSV file: '{args.clinical_file}'")

print(f"SEQUENCING on: {use_events}")

sequences = {}
datecols = {
    'interventi': 'date_interventi',
    'terapie': 'date_inizio_terapie',
    'vaccini_dosi': 'date_vaccini',
    'comorbidita': 'date_comorbidita',
    'eventi_inf_raggruppati': 'data_ev_inf',
    'eventi_tromb_raggruppati': 'data_ev_tromb',
    'plt_flwup_numeriche' : 'date_piastrine_flwup'
}

for id in tqdm(df_clinica.index.to_list(), desc="Sequencing"):
    sequences[id] = set()
    last_platelet = 'ND'
    plat_map = {'ND': 0, 'low':1, 'medium': 2, 'high':3}
    try:
        for event in use_events:
            if event == 'followup':
                dates = df_clinica.loc[id]['date_flwup']
                if not pd.isna(dates):
                    dlist = [d.strip() for d in dates.split(',') if not pd.isna(d.strip())]
                    for d in dlist:
                        sequences[id].add(('followup', str(pd.to_datetime(d).strftime('%Y-%m-%d')), 'followup', 'followup'))
                continue
            if event == 'splenectomizzato':
                ename = 'splenectomia' if args.lang == 'IT' else 'splenectomy'
                etype = 'intervento_chirurgico' if args.lang == 'IT' else 'surgical_operation'
                yes = 'SI' if args.lang == 'IT' else 'YES'
                if df_clinica.loc[id]['splenectomizzato'] == yes:
                #if df_clinica.loc[id]['splenectomizzato'] == 1:
                    d = df_clinica.loc[id]['data_splenectomia']
                    if not pd.isna(d):
                        sequences[id].add((ename, str(pd.to_datetime(d).strftime('%Y-%m-%d')), etype, ename))
                continue

            vals = df_clinica.loc[id][event]
            dates = df_clinica.loc[id][datecols[event]]

            if not pd.isna(vals) and not pd.isna(dates):
                elist = [e.strip() for e in vals.split(',')]
                dlist = [d.strip() for d in dates.split(',')]

                if event == 'comorbidita':
                    etype =  'comorbidità' if args.lang == 'IT' else 'comorbidity'
                    for x, d in zip(elist, dlist):
                        if not pd.isna(x) and not pd.isna(d):
                            sequences[id].add((comorbidity_map.get(x, 'altra comorbidità'), str(pd.to_datetime(d).strftime('%Y-%m-%d')), etype, x))

                elif event == 'eventi_inf_raggruppati':
                    etype = 'infezione' if args.lang == 'IT' else 'infection'
                    for x, d in zip(elist, dlist):
                        if not pd.isna(x) and not pd.to_datetime(d) is pd.NaT and not pd.isna(d):
                            sequences[id].add((infection_map.get(x, 'altra infezione'), str(pd.to_datetime(d).strftime('%Y-%m-%d')), etype, x))

                elif event == 'eventi_tromb_raggruppati':
                    etype = 'trombosi' if args.lang == 'IT' else 'thrombosys'
                    for x, d in zip(elist, dlist):
                        if not pd.isna(x) and not pd.to_datetime(d) is pd.NaT and not pd.isna(d):
                            sequences[id].add((x, str(pd.to_datetime(d).strftime('%Y-%m-%d')), etype, x))
                elif event == 'plt_flwup_numeriche':
                    etype = 'variazione_piastrine' if args.lang == 'IT' else 'platelet_change'
                    for x, d in zip(elist, dlist):
                        if not pd.isna(x) and not pd.isna(d):
                            if int(x) < 150000:     # calo piastrine
                                ename = 'decreased platelet count' if args.lang == 'IT' else 'decremento conteggio piastrine'
                                sequences[id].add((f'low', str(pd.to_datetime(d).strftime('%Y-%m-%d')), etype, ename))
                            elif int(x) >= 450000:  # aumento piastrine
                                ename = 'increased platelet count' if args.lang == 'IT' else 'incremento conteggio piastrine'
                                sequences[id].add((f'high', str(pd.to_datetime(d).strftime('%Y-%m-%d')), etype, ename))
                elif event == 'terapie':
                    etype = 'terapia' if args.lang == 'IT' else 'therapy'
                    for x, d in zip(elist, dlist):
                        if not pd.isna(x) and not pd.isna(d):
                            sequences[id].add((x, str(pd.to_datetime(d).strftime('%Y-%m-%d')), etype, x))
                elif event == 'vaccini_dosi':
                    etype = 'vaccinazione' if args.lang == 'IT' else 'vaccination'
                    for x, d in zip(elist, dlist):
                        if not pd.isna(x) and not pd.isna(d):
                            sequences[id].add((x, str(pd.to_datetime(d).strftime('%Y-%m-%d')), etype, x))
                elif event == 'interventi':
                    etype = 'intervento_chirurgico' if args.lang == 'IT' else 'surgical_operation'
                    for x, d in zip(elist, dlist):
                        if not pd.isna(x) and not pd.isna(d):
                            sequences[id].add((x, str(pd.to_datetime(d).strftime('%Y-%m-%d')), etype, x))
                else:
                    raise Exception("Unsupported event type!")

    except Exception as e:
        print(f'Patient {id}: Error on event "{event}" -> {e}')
        continue

    sequences[id] = sorted(list(sequences[id]), key=lambda x: x[1])

splenectomizzato
interventi
terapie
vaccini_dosi
comorbidita
eventi_inf_raggruppati
eventi_tromb_raggruppati
followup
plt_flwup_numeriche
SEQUENCING on: ['splenectomizzato', 'interventi', 'terapie', 'vaccini_dosi', 'comorbidita', 'eventi_inf_raggruppati', 'eventi_tromb_raggruppati', 'followup', 'plt_flwup_numeriche']


Sequencing:   0%|          | 0/1789 [00:00<?, ?it/s]

Patient 56: Error on event "splenectomizzato" -> NaTType does not support strftime


In [44]:
sequences[2]

[('anticoagulante', '2008-06-26', 'therapy', 'anticoagulante'),
 ('antibiotico', '2008-06-26', 'therapy', 'antibiotico'),
 ('splenectomy', '2008-06-26', 'surgical_operation', 'splenectomy'),
 ('followup', '2013-06-15', 'followup', 'followup')]

### BMI calculation from 1st follow_up
`df_minime`: dataset with a row contaning for each patient:
+ `bmi_num` : numeric BMI (weight / (heigth * height) )
+ `bmi` : categoric value for BMI
    - `underweight` if BMI < 18.5, 
    - `normal` if 18.5 < BMI <25 
    - `overweight` if 25 < BMI < 30
    - `obese` if BMI > 30
+ `date` : date of BMI recording

In [45]:
df0 = pd.read_csv(args.followup_file)

# define bins nd categories for BMI
#bins = [0, 18.5, 25, 30, np.inf]  vecchi range di INA
bins = [0, 20, 30, 40, np.inf]     # range da ICD-10-CM
labels = ['underweight', 'normal', 'overweight', 'obese']
# Convert 'data' column into datetime format
df0['data'] = pd.to_datetime(df0['creato_il'])
# find oldest (first in time) followup's date for each patient id
minime_per_id = df0.groupby('id_paziente')['data'].min().reset_index()
# merge original dataframe to get 
df_minime = df0.merge(minime_per_id, on=['id_paziente', 'data'], how='inner')
df_minime['bmi_num'] = np.nan  # inizializza la colonna
# select only rows with non-null weights and heights
mask = df_minime['peso'].notnull() & df_minime['altezza'].notnull()
# calculate numeri value for BMI
df_minime.loc[mask, 'bmi_num'] = df_minime.loc[mask, 'peso'] / ((df_minime.loc[mask, 'altezza']*0.01) ** 2)
# calculate categoric value for BMI
#df_minime['bmi'] = pd.cut(df_minime['bmi_num'], bins=bins, labels=labels, right=False)
df_minime['bmi'] = df_minime['bmi_num']
# Assegna la categoria corrispondente
df_minime.loc[mask]

,id,id_paziente,ido,ultimo_controllo,perso,id_periodicita,vivo,plt_medie,peso,altezza,id_decesso,causa_decesso,data_decesso,creato_da,creato_il,modificato_da,modificato_il,data,bmi_num,bmi
29,30,30,r,2015-11-23,0,13.0,1,NaN,98.0,168.0,NaN,NaN,NaN,41,2016-12-14 11:06:38,NaN,NaN,2016-12-14 11:06:38,34.722222,34.722222
30,31,31,r,2015-11-23,0,0.0,1,NaN,59.0,163.0,NaN,NaN,NaN,41,2016-12-14 11:39:38,NaN,NaN,2016-12-14 11:39:38,22.206331,22.206331
32,33,33,r,2015-11-16,0,0.0,1,NaN,82.0,171.0,NaN,NaN,NaN,41,2016-12-14 11:47:13,NaN,NaN,2016-12-14 11:47:13,28.042817,28.042817
34,35,35,r,2015-10-05,0,0.0,1,NaN,65.0,174.0,NaN,NaN,NaN,41,2016-12-14 12:10:34,NaN,NaN,2016-12-14 12:10:34,21.469150,21.469150
38,39,39,r,2015-05-25,0,0.0,1,NaN,36.0,146.0,NaN,NaN,NaN,41,2016-12-14 12:57:36,NaN,NaN,2016-12-14 12:57:36,16.888722,16.888722
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1879,7087,1944,NaN,2022-09-28,0,10.0,1,NaN,68.0,168.0,NaN,NaN,NaN,85,2022-09-28 15:49:04,NaN,NaN,2022-09-28 15:49:04,24.092971,24.092971
1880,7106,1945,NaN,2022-11-09,0,10.0,1,NaN,72.0,170.0,NaN,NaN,NaN,85,2022-11-09 16:26:13,NaN,NaN,2022-11-09 16:26:13,24.913495,24.913495
1884,7268,1947,NaN,2023-07-03,0,10.0,1,NaN,12.0,100.0,NaN,NaN,NaN,42,2023-07-03 09:57:34,NaN,NaN,2023-07-03 09:57:34,12.000000,12.000000
1890,7307,1951,NaN,2023-09-06,0,10.0,1,NaN,28.0,146.0,NaN,NaN,NaN,66,2023-09-27 17:12:10,NaN,NaN,2023-09-27 17:12:10,13.135673,13.135673


### Processing BMI change events from risk-records 

The `df1` dataset stores multiple rows for each patient, each one recording a new BMI numerical measure and the relative date.

In [46]:
df1 = pd.read_csv(args.risk_file)
df1.rename(columns={'bmi':'bmi_num'}, inplace=True)
df1['bmi'] = pd.cut(df1['bmi_num'], bins=bins, labels=labels, right=False)
df1['data'] = pd.to_datetime(df1['creato_il'])
df1[['id_paziente', 'bmi', 'bmi_num', 'data']]

,id_paziente,bmi,bmi_num,data
0,1926,underweight,18.166204,2022-01-17 18:42:01
1,1831,NaN,NaN,2022-01-20 12:56:59
2,1148,underweight,19.233561,2022-01-20 13:01:49
3,1149,underweight,19.140625,2022-01-20 13:11:46
4,1268,normal,23.011177,2022-01-26 14:56:44
...,...,...,...,...
306,1650,underweight,14.692378,2023-10-04 14:10:54
307,1954,underweight,14.541609,2023-10-09 12:05:27
308,1955,underweight,13.657648,2023-10-09 12:29:45
309,142,NaN,NaN,2023-10-24 11:07:54


This part of code detects only changes in BMI with respect to the more recent BMI measure which may be

- the one the first followup (in `df_minime`), or 
- a successive measure changing the previous cathegorical value (in `df1`).

In [47]:
if args.with_bmichange:
    # merge initial BMI measure to successive BMI measure
    df_all = pd.concat([df_minime, df1], ignore_index=True)

    # select only rows with no null date and BMI
    mask  = df_all['data'].notnull() & df_all['bmi'].notnull()
    df_all_valid = df_all.loc[mask]

    # Trova l'ultima misura valida per ogni paziente
    ultime_misure = df_all_valid.sort_values('data').groupby('id_paziente').tail(1)

    # merge df1 with dataframe of last valid BMI measure
    df1_con_ultime = df1.merge(
        ultime_misure[['id_paziente', 'data', 'bmi']],
        on='id_paziente',
        suffixes=('', '_ultima')
    )

    # Criteria for BMI-change event storing:
    # - date different from date of last valid BMI measure
    # - BMI measure different form last valid BMI measure
    # - both date and BMI measure are not null
    mask = (
        df1_con_ultime['data'].notna() &
        df1_con_ultime['bmi'].notna() &
        (df1_con_ultime['data'].dt.date != df1_con_ultime['data_ultima'].dt.date) &
        (df1_con_ultime['bmi'] != df1_con_ultime['bmi_ultima'])
    )
    # select only rows satisfying all criteria
    df1_filtrato = df1_con_ultime[mask].copy()
    # remove rows of same patient with eqaul dates (same day, may be different hours)
    df1_filtrato['data_giorno'] = df1_filtrato['data'].dt.date
    df1_filtrato = df1_filtrato.drop_duplicates(subset=['id_paziente', 'data_giorno'])
    # (Optional) remove temporary column
    df1_filtrato = df1_filtrato.drop(columns=['data_giorno'])
    df1_filtrato

## Savings JSON dataset
In this section we create in several steps the JSON dataset.

#### Translating functions

In [48]:
translator = json.load(open(args.translator_file))
def translate_event(e, trans_server):
    if e in translator.keys():
        en_e = translator[e]
    elif e == "splenectomia":
        en_e = "splenectomy"
    else:
        translator[e] = trans_server.translate(e).lower()
        en_e = translator[e]
    return en_e

def translate_icd_desc(e, trans_server):
    if e in translator.keys():
        en_e = translator[e]
    else:
        translator[e] = trans_server.translate(e).lower()
        en_e = translator[e]
    return en_e


def encode_event(etype, ename):
    e = ename.lower()
    if etype == "comorbidity":
        codetype = "ICD-10-CM"
        return (f'{icd_diseases_df.loc[e]["ICD10_Code"]}', icd_diseases_df.loc[e]["ICD10_Description_EN"], codetype) if e in icd_diseases_df.index else (None, None, None)
    elif etype == "infection":
        codetype = "ICD-10-CM"
        return (f'{icd_infections_df.loc[e]["ICD-10-CM"]}', icd_infections_df.loc[e]["Description"], codetype) if e in icd_infections_df.index else (None, None, None)
    elif etype == "therapy":
        codetype = "ICD-10-PCS"
        return (f'{icd_therapies_df.loc[e]["ICD10_PCS"]}', icd_therapies_df.loc[e]["Description"], codetype) if e in icd_therapies_df.index else (None, None, None)
    elif etype == "vaccination":
        codetype = "CPT"
        return (f'{icd_vaccines_df.loc[e]["CPT_Code"]}', icd_vaccines_df.loc[e]["Description"], codetype) if e in icd_vaccines_df.index else (None, None, None)
    elif etype == "surgical_operation":
        codetype = "ICD-10-PCS"
        return (f'{icd_procedures_df.loc[e]["ICD-10-PCS"]}', icd_procedures_df.loc[e]["ICD10_Description_EN"], codetype) if e in icd_procedures_df.index else (None, None, None)
    elif etype == "bmi_change":
        codetype = "ICD-10_CM"
        if e == "underweight": 
            code, code_descr = "Z68.1", "Body mass index [BMI] 19.9 or less, adult"
        elif e == "normal":
            code, code_descr = "Z68.2", "Body mass index [BMI] 20-29, adult"
        elif e == "overweight":
            code, code_descr = "Z68.3", "Body mass index [BMI] 30-39, adult"
        elif e == "obese":
            code, code_descr = "Z68.4", "Body mass index [BMI] 40 or greater, adult"
        else:
            code, code_descr =  None, None
        return codetype, code, code_descr
    elif etype == "platelet_change":
        codetype = "ICD-10_CM"
        if e == "low": 
            code, code_descr = "D69.6", "Thrombocytopenia, unspecified"
        elif e == "high":
            code, code_descr = "D75.83", "Thrombocytosis, unspecified"
        else:
            code, code_descr =  None, None
        return codetype, code, code_descr
    elif etype == "thrombosys":
        codetype = "ICD-10_CM"
        return (f'{icd_diseases_df.loc[e]["ICD10_Code"]}', icd_diseases_df.loc[e]["ICD10_Description_EN"], codetype) if e in icd_diseases_df.index else (None, None, None)
    else:
        return None, None, None
    

### merge events and BMI changes
Now that we have sequences of events from clinical EHRs and we have defined new events for BMI changes, we put everything together.

In [49]:
from translate import Translator
trans_srv = Translator(from_lang='it', to_lang="en")
trans_srv_to_it = Translator(from_lang='en', to_lang="it")
if args.lang == "EN":
    df_events = pd.DataFrame(columns=['id', 'type', 'event', 'date', 'descr', 'code', 'code_descr', 'code_type'])
else:
    df_events = pd.DataFrame(columns=['id', 'tipo', 'evento', 'data', 'descr', 'codice', 'codice_descr', 'codice_tipo'])
recorded_attributes = list(pd.read_csv(args.static_attributes_file, header=None, comment='#')[0].values) + ['data_nascita']
for id, seq in tqdm(sequences.items(), desc="Event DT"):
    newseq = seq.copy()
    if id in df1_filtrato.index:  # se id_paziente è nel dataframe dei cambi bmi
        bmi_change = 'overweight' if df1_filtrato.loc[id]['bmi'] < df1_filtrato.loc[id]['bmi_ultima'] else 'underweight'
        bmi_chg_status = 'bmi incresae' if df1_filtrato.loc[id]['bmi'] < df1_filtrato.loc[id]['bmi_ultima'] else 'bmi decrease'
        newseq += [(bmi_change, str(df1_filtrato.loc[id]['data_ultima'].strftime('%Y-%m-%d')), 'bmi_change', 'bmi_chg_status')]
        newseq = sorted(newseq, key=lambda x: x[1])
    for e in newseq:
        code, code_desc, code_type = encode_event(e[2], e[3])
        #print(e[2], e[3], code_desc)
        df_events = pd.concat([df_events, pd.DataFrame([[id,          # id
                                                         e[2],                                                          # type (already in chosen language)
                                                         e[0] if args.lang == 'IT' else translate_event(e[0], trans_srv),    # event name by INA 
                                                         e[1],                                                          # date
                                                         e[3] if args.lang == 'IT' else translate_event(e[3], trans_srv),    # event description by INA
                                                         code,                          # event code by Standard (ICD-10, HPCPS or CPT)
                                                         translate_icd_desc(code_desc, trans_srv_to_it) if code_desc is not None and args.lang == 'IT' else code_desc,                     
                                                                              # event description by Standard (ICD-10, HPCPS or CPT)
                                                         code_type                      # code type (ICD-10-CM, ICD-10_PCS, HPCPS or CPT)
                                                         ]], 
                                                        columns=df_events.columns)], ignore_index=True)

dfc = df_clinica[recorded_attributes + ['eventi_infettivi']]
dfc = pd.concat([df_clinica[recorded_attributes], df_clinica['eventi_infettivi']], axis=1)
if args.lang == 'EN':
    dfc.replace(translator, inplace=True)
dfc.replace({np.nan:None}, inplace=True)
if args.lang == 'EN':
    dfc.rename(columns=translator, inplace=True)
df_events['type' if args.lang == 'EN' else 'tipo'].value_counts()


Event DT:   0%|          | 0/1789 [00:00<?, ?it/s]

type
vaccination           8074
followup              6571
therapy               6252
surgical_operation    1583
platelet_change       1411
comorbidity           1087
infection              807
thrombosys             152
Name: count, dtype: int64

In [50]:
df_events

,id,type,event,date,descr,code,code_descr,code_type
0,1,therapy,antiplatelet aggregation,2001-10-31,antiplatelet aggregation,3E03317,Introduction of antiplatelet agent,ICD-10-PCS
1,1,therapy,antibiotic,2001-10-31,antibiotic,3E03329,Introduction of anti-infective agent,ICD-10-PCS
2,1,surgical_operation,splenectomy,2001-10-31,splenectomy,07TP0ZZ,"Resection of spleen, open approach",ICD-10-PCS
3,1,therapy,anticoagulant,2001-10-31,anticoagulant,3E03317,Introduction of anticoagulant agent,ICD-10-PCS
4,1,vaccination,vaccine Pneumo23,2009-12-04,vaccine Pneumo23,90732,"Pneumococcal polysaccharide vaccine, 23-valent...",CPT
...,...,...,...,...,...,...,...,...
25932,1952,vaccination,vaccine meningococcus B,2023-06-30,vaccine meningococcus B,90620,"Meningococcal B vaccine, 2-dose schedule",CPT
25933,1952,vaccination,vaccine meningococcus B,2023-07-30,vaccine meningococcus B,90620,"Meningococcal B vaccine, 2-dose schedule",CPT
25934,1952,vaccination,vaccine Pneumo23,2023-08-14,vaccine Pneumo23,90732,"Pneumococcal polysaccharide vaccine, 23-valent...",CPT
25935,1952,surgical_operation,splenectomy,2023-09-05,splenectomy,07TP0ZZ,"Resection of spleen, open approach",ICD-10-PCS


### Update translator

In [51]:
import json 
with open(args.translator_file, "w") as f:
    json.dump(translator, f, indent=2)


#### Create dataset
Static information is joined to the event sequence for each patient in a dictionary. 
```
{"id": <patient_id>,
  ... static info
  "events": [{"type": <event_type> , "event": <event_name> , "date": <event_date> , "descr": <description>}
             ...]
}
```
The list of dictionaries for the patients is returned.

In [52]:
dataset = []
import math

def nan2None(x):
    if isinstance(x, float) and math.isnan(x):
        return None
    else:
        return x
for index, row in tqdm(dfc.iterrows(), desc='JSON data', total=len(dfc)):
    persona = row.to_dict()
    persona['id'] = index
    persona = {k:nan2None(v) for k,v in persona.items()}
    if index in df_minime['id_paziente'].unique() and not pd.isna(df_minime[df_minime['id_paziente'] == index]['bmi'].values[0]):
        persona['bmi'] = df_minime[df_minime['id_paziente'] == index]['bmi'].values[0]
    ev = df_events[df_events['id'] == index]
    if args.lang == 'EN':
        persona['events'] = ev[['type', 'event', 'date', 'descr', 'code', 'code_descr', 'code_type']].to_dict(orient='records')
    else:
        persona['eventi'] = ev[['tipo', 'evento', 'data', 'descr', 'codice', 'codice_descr', 'codice_tipo']].to_dict(orient='records')
    dataset.append(persona)

JSON data:   0%|          | 0/1789 [00:00<?, ?it/s]

## Time shift

In [53]:
if args.with_timeshift:
    import datetime
    mindate = datetime.datetime.min
    maxdate = datetime.datetime.max
    for person in tqdm(dataset, desc='Get range', total=len(dataset)):
        EHRs = person['events']
        for record in EHRs:
            if record['type'] == "followup":
                date = pd.to_datetime(record['date'])
                if date > mindate:
                    mindate = date
                if date < maxdate:
                    maxdate = date
    maxdate, mindate

Get range:   0%|          | 0/1789 [00:00<?, ?it/s]

### Shift dates randomly of 100 years

In [54]:
if args.with_timeshift:
    from random import randrange
    import numpy as np
    
    for person in tqdm(dataset, desc='Time shift', total=len(dataset)):
        bdate_field = 'birth_date' if args.lang == 'EN' else 'data_nascita'
        date_field = 'date' if args.lang == 'EN' else 'data'
        events_field = 'events' if args.lang == 'EN' else 'eventi'
        deltatime = pd.Timedelta(days=randrange(36500,54750))
        newdate = pd.to_datetime(person[bdate_field])
        newdate += deltatime
        person[bdate_field] = str(newdate.strftime('%Y-%m-%d'))
        EHRs = person[events_field]
        for record in EHRs:
            newdate = pd.to_datetime(record[date_field]) + deltatime
            record[date_field] = str(newdate.strftime('%Y-%m-%d'))

Time shift:   0%|          | 0/1789 [00:00<?, ?it/s]

## Age Group

In [55]:
from datetime import datetime
def get_age(age, lang):
    if age == np.inf:
        return None
    else:
        if lang == "IT":
            return 'pediatrico' if age <= 12 else 'adolescente' if age < 18 else "giovane" if age < 35 else "maturo" if age < 55 else "anziano" if age < 75 else "geriatrico"
        else:
            return 'pediatric' if age <= 12 else 'adolescent' if age < 18 else "young" if age < 35 else "mature" if age < 55 else "elder" if age < 75 else "geriatric"

def days_between(d1, d2):
    return round((d2 - d1).days / 365, 2)

if args.lang == "EN":
    for data in tqdm(dataset, desc="Age group"):
        birth_date = datetime.strptime(data["birth_date"], "%Y-%m-%d") if data["birth_date"] else "0000-00-00"
        # Find the splenectomy date
        splenectomy_dates = [datetime.strptime(e["date"], "%Y-%m-%d") for e in data["events"] if e["event"] == "splenectomy"]
        followup_dates = [datetime.strptime(e["date"], "%Y-%m-%d") for e in data["events"] if e["type"] == "followup"]
        splenectomy_date = splenectomy_dates[0] if splenectomy_dates else None
        followup_date = followup_dates[0] if followup_dates else None
        if splenectomy_date:
            age_spleen = days_between(birth_date, splenectomy_date)
            data['age_at_splenectomy'] = get_age(age_spleen, args.lang)
        else:
            data['age_at_splenectomy'] = None
        if followup_date:
            age_flwp = days_between(birth_date, followup_date)
            data['age_group'] = get_age(age_flwp, args.lang)
        else:
            data['age_group'] = None
elif args.lang == "IT":
    for data in tqdm(dataset, desc="Age group"):
        birth_date = datetime.strptime(data["data_nascita"], "%Y-%m-%d") if data["data_nascita"] else "0000-00-00"
        # Find the splenectomy date
        splenectomy_dates = [datetime.strptime(e["data"], "%Y-%m-%d") for e in data["eventi"] if e["evento"] == "splenectomia"]
        followup_dates = [datetime.strptime(e["date"], "%Y-%m-%d") for e in data["eventi"] if e["tipo"] == "followup"]
        splenectomy_date = splenectomy_dates[0] if splenectomy_dates else None
        followup_date = followup_dates[0] if followup_dates else None
        if splenectomy_date:
            age_spleen = days_between(birth_date, splenectomy_date)
            data['eta_alla_splnectomia'] = get_age(age_spleen, args.lang)
        else:
            data['eta_alla_splnectomia'] = None
            age_flwp = days_between(birth_date, followup_date)
        if followup_date:
            data['eta'] = get_age(age_flwp, args.lang)
        else:
            data['eta'] = None


Age group:   0%|          | 0/1789 [00:00<?, ?it/s]

## Derive variable counters

In [56]:
import datetime
if args.with_qt:
    from datetime import datetime

    if args.lang == "EN":
        for data in tqdm(dataset, desc="Derived qt"):
            count_pre_flu = count_post_flu = count_pre_hib = count_post_hib = count_pre_pneumo = count_post_pneumo = count_pre_pcv13 = count_post_pcv13 = count_pre_meningo = count_post_meningo = 0
            min_days_hib = min_days_pneumo = min_days_meningo = min_days_flu = min_days_pcv13 = np.inf
            for e in data["events"]:
                event_date = datetime.strptime(e["date"], "%Y-%m-%d")
                # Count vaccinations before splenectomy
                if e["type"] == "vaccination":
                    if e["event"] == "vaccine Hib":
                        if splenectomy_date:
                            if event_date <= splenectomy_date:
                                count_pre_hib += 1
                            else:
                                count_post_hib += 1
                        min_days_hib = min(min_days_hib, days_between(birth_date, event_date))
                    elif e["event"] == "vaccine Pneumo23":
                        if splenectomy_date:
                            if event_date <= splenectomy_date:
                                count_pre_pneumo += 1
                            else:
                                count_post_pneumo += 1
                        min_days_pneumo = min(min_days_pneumo, days_between(birth_date, event_date))
                    elif e["event"] == "vaccine PCV13":
                        if splenectomy_date:
                            if event_date <= splenectomy_date:
                                count_pre_pcv13 += 1
                            else:
                                count_post_pcv13 += 1
                        min_days_pcv13 = min(min_days_pcv13, days_between(birth_date, event_date))
                    elif e["event"] == "vaccine meningococcus ACWY" or e["event"] == "vaccine meningococcus B" or e["event"] == "vaccine meningococcus C":
                        if splenectomy_date:
                            if event_date <= splenectomy_date:
                                count_pre_meningo += 1
                            else:
                                count_post_meningo += 1
                        min_days_meningo = min(min_days_meningo, days_between(birth_date, event_date))
                    elif e["event"] == "vaccine flu":
                        if splenectomy_date:
                            if event_date <= splenectomy_date:
                                count_pre_flu += 1
                            else:
                                count_post_flu += 1
                        min_days_flu = min(min_days_flu, days_between(birth_date, event_date))
                    else:
                        continue
            data['age_vax_pcv13'] = get_age(min_days_pcv13, args.lang)
            data['age_vax_hib'] = get_age(min_days_hib, args.lang)
            data['age_vax_pneumo'] = get_age(min_days_pneumo, args.lang)
            data['age_vax_meningo'] = get_age(min_days_meningo, args.lang)
            data['age_vax_flu'] = get_age(min_days_flu, args.lang)
            data['pre_splenectomy_vax_pcv13'] = 'YES' if count_pre_pcv13 > 0 else 'NO'
            data['pre_splenectomy_vax_hib'] = 'YES' if count_pre_hib > 0 else 'NO'
            data['pre_splenectomy_vax_pneumo'] =  'YES' if count_pre_pneumo > 0 else 'NO'
            data['pre_splenectomy_vax_meningo'] =  'YES' if count_pre_meningo > 0 else 'NO'
            data['pre_splenectomy_vax_flu'] =  'YES' if count_pre_flu > 0 else 'NO'
            data['post_splenectomy_vax_pcv13_cnt'] = count_post_pcv13
            data['post_splenectomy_vax_hib_cnt'] = count_post_hib
            data['post_splenectomy_vax_pneumo_cnt'] = count_post_pneumo
            data['post_splenectomy_vax_meningo_cnt'] = count_post_meningo
            data['post_splenectomy_vax_flu_cnt'] = count_post_flu
            data['qt_vax_profilassi'] = count_pre_hib + count_pre_pneumo + count_pre_meningo + count_pre_pcv13 + count_pre_flu
            data['qt_vax_post_profilassi'] = count_post_hib + count_post_pneumo + count_post_meningo + count_post_pcv13 + count_post_flu
            data['qt_vax_totali'] = data['qt_vax_profilassi'] + data['qt_vax_post_profilassi'] 
    else:
        for data in tqdm(dataset, desc="Deriva qt"):
            count_pre_flu = count_post_flu = count_pre_hib = count_post_hib = count_pre_pneumo = count_post_pneumo = count_pre_pcv13 = count_post_pcv13 = count_pre_meningo = count_post_meningo = 0
            min_days_hib = min_days_pneumo = min_days_meningo = min_days_flu = min_days_pcv13 = np.inf
            for e in data["eventi"]:
                event_date = datetime.strptime(e["data"], "%Y-%m-%d")
                # Count vaccinations before splenectomy
                if e["tipo"] == "vaccinazione":
                    if e["evento"] == "vaccino_hib":
                        if splenectomy_date:
                            if event_date <= splenectomy_date:
                                count_pre_hib += 1
                            else:
                                count_post_hib += 1
                        min_days_hib = min(min_days_hib, days_between(birth_date, event_date))
                    elif e["evento"] == "vaccino_pneumo23":
                        if splenectomy_date:
                            if event_date <= splenectomy_date:
                                count_pre_pneumo += 1
                            else:
                                count_post_pneumo += 1
                        min_days_pneumo = min(min_days_pneumo, days_between(birth_date, event_date))
                    elif e["evento"] == "vaccino_pcv13":
                        if splenectomy_date:
                            if event_date <= splenectomy_date:
                                count_pre_pcv13 += 1
                            else:
                                count_post_pcv13 += 1
                        min_days_pcv13 = min(min_days_pcv13, days_between(birth_date, event_date))
                    elif e["evento"] == "vaccino_meningococco_acwy" or e["evento"] == "vaccino_meningococco_b" or e["evento"] == "vaccino_meningococco_c":
                        if splenectomy_date:
                            if event_date <= splenectomy_date:
                                count_pre_meningo += 1
                            else:
                                count_post_meningo += 1
                        min_days_meningo = min(min_days_meningo, days_between(birth_date, event_date))
                    elif e["evento"] == "vaccino_influenza":
                        if splenectomy_date:
                            if event_date <= splenectomy_date:
                                count_pre_flu += 1
                            else:
                                count_post_flu += 1
                        min_days_flu = min(min_days_flu, days_between(birth_date, event_date))
                    else:
                        continue
            data['eta_vax_pcv13'] = get_age(min_days_pcv13, args.lang)
            data['eta_vax_hib'] = get_age(min_days_hib, args.lang)
            data['eta_vax_pneumo'] = get_age(min_days_pneumo, args.lang)
            data['eta_vax_meningo'] = get_age(min_days_meningo, args.lang)
            data['eta_vax_influenza'] = get_age(min_days_flu, args.lang)
            data['profilassi_vax_pcv13'] = 'SI' if count_pre_pcv13 > 0 else 'NO'
            data['profilassi_vax_hib'] = 'SI' if count_pre_hib > 0 else 'NO'
            data['profilassi_vax_pneumo'] =  'SI' if count_pre_pneumo > 0 else 'NO'
            data['profilassi_vax_meningo'] =  'SI' if count_pre_meningo > 0 else 'NO'
            data['profilassi_vax_influenza'] =  'SI' if count_pre_flu > 0 else 'NO'
            data['post_splenectomia_vax_pcv13_cnt'] = count_post_pcv13
            data['post_splenectomia_vax_hib_cnt'] = count_post_hib
            data['post_splenectomia_vax_pneumo_cnt'] = count_post_pneumo
            data['post_splenectomia_vax_meningo_cnt'] = count_post_meningo
            data['post_splenectomia_vax_flu_cnt'] = count_post_flu
            data['qt_vax_profilassi'] = count_pre_hib + count_pre_pneumo + count_pre_meningo + count_pre_pcv13 + count_pre_flu
            data['qt_vax_post_profilassi'] = count_post_hib + count_post_pneumo + count_post_meningo + count_post_pcv13 + count_post_flu
            data['qt_vax_totali'] = data['qt_vax_profilassi'] + data['qt_vax_post_profilassi'] 


## Save dataset to JSON

In [57]:
import json
import os
suffix = f"_{args.flwup_nth}flw" if args.flwup_nth != np.inf else ""
suffix += "_1inf" if args.stop_at_1st_inf else ""
suffix += f"_{args.lang}"
filename = args.dataset_file + suffix + "_newicd.json"
with open(filename, "w") as f:
    json.dump(dataset, f, indent=2)

print(f"✅ Dataset completo (statici + eventi) salvato in '{filename}'")
print(len(dfc))


✅ Dataset completo (statici + eventi) salvato in 'dataset_14_04_26_EN_newicd.json'
1789


### Save to CSV

In [58]:
static_keys = list(set(list(dataset[0].keys())) - set(["events"]))
df_static = pd.DataFrame(columns=static_keys)
for data in tqdm(dataset, desc="JSON dataset tp CSV:"):
    df_static = pd.concat([df_static, pd.DataFrame([[data[k] for k in static_keys]], columns=df_static.columns)], ignore_index=True)
df_events.to_csv(args.events_file + suffix + "_ccs.csv", index=False)
df_static.to_csv(args.data_file+ suffix + "_ccs.csv")

JSON dataset tp CSV::   0%|          | 0/1789 [00:00<?, ?it/s]

In [59]:
df_clinica.columns

Index(['id_centro', 'sesso', 'data_nascita', 'tipo_osservazione',
       'data_inizio_prospettica', 'area_pat_base', 'patologia_base',
       'genotipo_beta1', 'genotipo_beta2', 'genotipo_alfa1', 'genotipo_alfa2',
       'hbf', 'qt_schede_paziente', 'data_flwup', 'età_flwup', 'qt_follow_up',
       'date_flwup', 'date_piastrine_flwup', 'piastrine_schede_flwup',
       'plt_flwup_numeriche', 'piastrine_flwup_medie', 'vivo', 'causa_decesso',
       'data_decesso', 'perso', 'splenectomizzato', 'data_splenectomia',
       'indicazione_splenectomia', 'risposta_splenectomia',
       'modalita_splenectomia', 'milza_accessoria', 'eparina',
       'eparina_farmaco', 'eparina_dosaggio_num', 'eparina_gg_prima',
       'eparina_gg_dopo', 'profilassi_vax_hib', 'profilassi_vax_pneumo',
       'profilassi_vax_meningo', 'profilassi_vax_influenza',
       'profilassi_vax_covid', 'age_group', 'diff_nascita_splen',
       'diff_morte_intervento', 'diff_vax_hib_splen', 'diff_vax_pneumo_splen',
       'dif